# Logistic Regression Term Prediction

This notebook trains **PCA + Logistic Regression** models for predicting CogAtlas terms from brain activation patterns.

We train on 4 different datasets:
1. **CogAtlas only** (~808 terms): Concepts, disorders, and tasks from CogAtlas
2. **Merged Complete** (~1,519 terms): CogAtlas + all brain region n-grams
3. **Coarse Hierarchical** (~825 terms): CogAtlas + 17 broad brain regions (e.g., "frontal lobe")
4. **Intermediate Hierarchical** (variable): CogAtlas + ~50-100 intermediate brain regions (e.g., "dorsolateral prefrontal cortex")

**Input**: Projected brain vectors (384-dim from InfoNCE projection head)

**Output**: Multi-label predictions for each term

**Note**: The coarse hierarchical model (Dataset 3) performs poorly due to extreme class imbalance (65.5% of images have "frontal lobe"). The intermediate hierarchical model (Dataset 4) addresses this with finer-grained categories.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append('../../')

from src.neurovlm.term_classification_models import (
    load_and_align_term_data,
    LogisticTermPredictor,
    evaluate_term_prediction,
    print_term_evaluation_results
)
from sklearn.model_selection import train_test_split

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Dataset 1: CogAtlas Only (~808 terms)

In [ ]:
print("="*70)
print("DATASET 1: CogAtlas Only")
print("="*70)

# Load data
X_cog, y_cog, pmids_cog, terms_cog = load_and_align_term_data(
    data_source='cogatlas'
)

# Split
X_train_cog, X_test_cog, y_train_cog, y_test_cog = train_test_split(
    X_cog, y_cog, test_size=0.2, random_state=42
)

print(f"\nTrain: {X_train_cog.shape}, Test: {X_test_cog.shape}")

In [7]:
# Train model
model_cog = LogisticTermPredictor(
    C=1.0,
    class_weight='balanced',
    random_state=42
)

print("Training Logistic Regression on CogAtlas data...")
model_cog.fit(X_train_cog, y_train_cog, verbose=True)

Training Logistic Regression on CogAtlas data...
Scaling features...
Training multi-output logistic regression...
  Input shape: (21112, 384)
  Output shape: (21112, 814)
Training complete!


In [8]:
# Evaluate
y_pred_cog = model_cog.predict(X_test_cog)
y_proba_cog = model_cog.predict_proba(X_test_cog)

results_cog = evaluate_term_prediction(
    y_test_cog, y_pred_cog, y_proba_cog, terms_cog, top_k=10
)

print("\n=== CogAtlas Model Results ===")
print_term_evaluation_results(results_cog)


=== CogAtlas Model Results ===

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.145      0.134
Recall                              0.690      0.648
F1                                  0.240      0.217

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.2234
  coverage_error: 465.9193
  label_ranking_loss: 0.2250

=== Top-10 Accuracy ===
  0.533 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.0345 - Average fraction of true labels found in top-5
  recall@7: 0.0498 - Average fraction of true labels found in top-7
  recall@10: 0.0711 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.7935
  Macro-AUC: 0.7706

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F1    Support
------------------------------------------

### Without pca

In [3]:
from neurovlm.term_classification_models import LogisticTermPredictor

model = LogisticTermPredictor(
  n_terms=808,
  C=1.0,
  class_weight='balanced',
  random_state=42
)
model.fit(X_train_cog, y_train_cog)



Scaling features...
Scaled feature shape: (23894, 384)
Training multi-output logistic regression for 808 terms...
Training complete!


In [4]:
# Evaluate
y_pred_cog = model.predict(X_test_cog)
y_proba_cog = model.predict_proba(X_test_cog)

results_cog = evaluate_term_prediction(
    y_test_cog, y_pred_cog, y_proba_cog, terms_cog, top_k=10
)

print("\n=== CogAtlas Model Results ===")
print_term_evaluation_results(results_cog)


=== CogAtlas Model Results ===

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.509      0.422
Recall                              0.854      0.724
F1                                  0.638      0.528

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.8016
  coverage_error: 39.7466
  label_ranking_loss: 0.0063

=== Top-10 Accuracy ===
  1.000 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.3721 - Average fraction of true labels found in top-5
  recall@7: 0.4947 - Average fraction of true labels found in top-7
  recall@10: 0.6448 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9930
  Macro-AUC: 0.9886
  AUC-ROC scores for multi-label classification

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F1    Sup

## Dataset 2: Merged Complete (~1,519 terms)

In [ ]:
print("\n" + "="*70)
print("DATASET 2: Merged Complete (CogAtlas + Brain Regions)")
print("="*70)

# Load data
X_merged, y_merged, pmids_merged, terms_merged = load_and_align_term_data(
    data_source='merged_complete_term'
)

# Split
X_train_merged, X_test_merged, y_train_merged, y_test_merged = train_test_split(
    X_merged, y_merged, test_size=0.2, random_state=42
)

print(f"\nTrain: {X_train_merged.shape}, Test: {X_test_merged.shape}")

In [6]:
# Train model
model_merged = PCATermPredictor(
    n_terms=y_merged.shape[1],
    variance_threshold=0.95,
    C=1.0,
    random_state=42
)

print("Training PCA + Logistic Regression on Merged Complete data...")
model_merged.fit(X_train_merged, y_train_merged, verbose=True)

Training PCA + Logistic Regression on Merged Complete data...
Scaling features...
Fitting PCA...
Using 120 components (95.0% variance)
PCA shape: (23894, 120)
Training multi-output logistic regression for 1519 terms...
Training complete!


In [7]:
# Evaluate
y_pred_merged = model_merged.predict(X_test_merged)
y_proba_merged = model_merged.predict_proba(X_test_merged)

results_merged = evaluate_term_prediction(
    y_test_merged, y_pred_merged, y_proba_merged, terms_merged, top_k=10
)

print("\n=== Merged Complete Model Results ===")
print_term_evaluation_results(results_merged)


=== Merged Complete Model Results ===

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.748      0.333
Recall                              0.370      0.280
F1                                  0.495      0.301

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.5759
  coverage_error: 547.2025
  label_ranking_loss: 0.0565

=== Top-10 Accuracy ===
  1.000 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.2383 - Average fraction of true labels found in top-5
  recall@7: 0.3172 - Average fraction of true labels found in top-7
  recall@10: 0.4093 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9376
  Macro-AUC: 0.7613
  AUC-ROC scores for multi-label classification

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F

## Dataset 3: Hierarchical Brain Regions

In [ ]:
print("\n" + "="*70)
print("DATASET 3: Hierarchical Brain Regions")
print("="*70)

# Load data
X_hier, y_hier, pmids_hier, terms_hier = load_and_align_term_data(
    data_source='hierarchical_term'
)

# Split
X_train_hier, X_test_hier, y_train_hier, y_test_hier = train_test_split(
    X_hier, y_hier, test_size=0.2, random_state=42
)

print(f"\nTrain: {X_train_hier.shape}, Test: {X_test_hier.shape}")

In [9]:
# Train model
model_hier = PCATermPredictor(
    n_terms=y_hier.shape[1],
    variance_threshold=0.95,
    C=1.0,
    random_state=42
)

print("Training PCA + Logistic Regression on Hierarchical data...")
model_hier.fit(X_train_hier, y_train_hier, verbose=True)

Training PCA + Logistic Regression on Hierarchical data...
Scaling features...
Fitting PCA...
Using 120 components (95.0% variance)
PCA shape: (23894, 120)
Training multi-output logistic regression for 825 terms...
Training complete!


In [10]:
# Evaluate
y_pred_hier = model_hier.predict(X_test_hier)
y_proba_hier = model_hier.predict_proba(X_test_hier)

results_hier = evaluate_term_prediction(
    y_test_hier, y_pred_hier, y_proba_hier, terms_hier, top_k=10
)

print("\n=== Hierarchical Model Results ===")
print_term_evaluation_results(results_hier)


=== Hierarchical Model Results ===

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.739      0.615
Recall                              0.570      0.517
F1                                  0.644      0.555

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.7796
  coverage_error: 49.8753
  label_ranking_loss: 0.0078

=== Top-10 Accuracy ===
  1.000 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.3139 - Average fraction of true labels found in top-5
  recall@7: 0.4183 - Average fraction of true labels found in top-7
  recall@10: 0.5464 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9911
  Macro-AUC: 0.9805
  AUC-ROC scores for multi-label classification

=== Performance on Most Common Terms ===
Term                                      Precision     Recall         F1   

## Dataset 4: Intermediate Hierarchical Brain Regions

In [ ]:
print("\n" + "="*70)
print("DATASET 4: Intermediate Hierarchical Brain Regions")
print("="*70)

# Load data
X_inter, y_inter, pmids_inter, terms_inter = load_and_align_term_data(
    data_source='fine_hierarchical_term'
)

# Split
X_train_inter, X_test_inter, y_train_inter, y_test_inter = train_test_split(
    X_inter, y_inter, test_size=0.2, random_state=42
)

print(f"\nTrain: {X_train_inter.shape}, Test: {X_test_inter.shape}")

In [9]:
# Train model
model_inter = PCATermPredictor(
    n_terms=y_inter.shape[1],
    variance_threshold=0.95,
    C=1.0,
    random_state=42
)

print("Training PCA + Logistic Regression on Intermediate Hierarchical data...")
model_inter.fit(X_train_inter, y_train_inter, verbose=True)

Training PCA + Logistic Regression on Intermediate Hierarchical data...
Scaling features...
Fitting PCA...
Using 120 components (95.0% variance)
PCA shape: (23894, 120)
Training multi-output logistic regression for 898 terms...
Training complete!


In [10]:
# Evaluate
y_pred_inter = model_inter.predict(X_test_inter)
y_proba_inter = model_inter.predict_proba(X_test_inter)

results_inter = evaluate_term_prediction(
    y_test_inter, y_pred_inter, y_proba_inter, terms_inter, top_k=10
)

print("\n=== Intermediate Hierarchical Model Results ===")
print_term_evaluation_results(results_inter)


=== Intermediate Hierarchical Model Results ===

=== Overall Performance ===
Metric                              Micro      Macro
----------------------------------------------------
Precision                           0.748      0.564
Recall                              0.520      0.474
F1                                  0.614      0.509

=== Ranking Metrics ===
  label_ranking_avg_precision: 0.7327
  coverage_error: 89.3418
  label_ranking_loss: 0.0144

=== Top-10 Accuracy ===
  1.000 - Fraction of samples with at least one true term in top-10

=== Recall@K Metrics ===
  recall@5: 0.3171 - Average fraction of true labels found in top-5
  recall@7: 0.4217 - Average fraction of true labels found in top-7
  recall@10: 0.5443 - Average fraction of true labels found in top-10

=== AUC Metrics ===
  Micro-AUC: 0.9839
  Macro-AUC: 0.9412
  AUC-ROC scores for multi-label classification

=== Performance on Most Common Terms ===
Term                                      Precision     Recall 

## Comparison Across Datasets

In [ ]:
# Create comparison table
comparison = pd.DataFrame([
    {
        'Dataset': 'CogAtlas Only',
        'Num Terms': len(terms_cog),
        'Micro F1': results_cog['micro']['f1'],
        'Macro F1': results_cog['macro']['f1'],
        'Recall@5': results_cog['recall_at_k']['recall@5'],
        'Recall@10': results_cog['recall_at_k']['recall@10'],
        'Label Ranking AP': results_cog['ranking']['label_ranking_avg_precision']
    },
    {
        'Dataset': 'Merged Complete',
        'Num Terms': len(terms_merged),
        'Micro F1': results_merged['micro']['f1'],
        'Macro F1': results_merged['macro']['f1'],
        'Recall@5': results_merged['recall_at_k']['recall@5'],
        'Recall@10': results_merged['recall_at_k']['recall@10'],
        'Label Ranking AP': results_merged['ranking']['label_ranking_avg_precision']
    },
    {
        'Dataset': 'Coarse Hierarchical',
        'Num Terms': len(terms_hier),
        'Micro F1': results_hier['micro']['f1'],
        'Macro F1': results_hier['macro']['f1'],
        'Recall@5': results_hier['recall_at_k']['recall@5'],
        'Recall@10': results_hier['recall_at_k']['recall@10'],
        'Label Ranking AP': results_hier['ranking']['label_ranking_avg_precision']
    },
    {
        'Dataset': 'Intermediate Hierarchical',
        'Num Terms': len(terms_inter),
        'Micro F1': results_inter['micro']['f1'],
        'Macro F1': results_inter['macro']['f1'],
        'Recall@5': results_inter['recall_at_k']['recall@5'],
        'Recall@10': results_inter['recall_at_k']['recall@10'],
        'Label Ranking AP': results_inter['ranking']['label_ranking_avg_precision']
    }
])

print("\n" + "="*70)
print("LOGISTIC REGRESSION: Dataset Comparison")
print("="*70)
print(comparison.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Micro F1', 'Macro F1', 'Label Ranking AP']
for i, metric in enumerate(metrics):
    axes[i].bar(comparison['Dataset'], comparison[metric], alpha=0.7, edgecolor='black')
    axes[i].set_ylabel(metric)
    axes[i].set_title(f'{metric} Comparison')
    axes[i].set_ylim([0, 1])
    axes[i].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for j, v in enumerate(comparison[metric]):
        axes[i].text(j, v + 0.02, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

## Save Models

In [ ]:
import pickle

# Save models
output_dir = Path('saved_models/logistic_regression')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'cogatlas_model.pkl', 'wb') as f:
    pickle.dump(model_cog, f)
print(f"Saved CogAtlas model to {output_dir / 'cogatlas_model.pkl'}")

with open(output_dir / 'merged_complete_model.pkl', 'wb') as f:
    pickle.dump(model_merged, f)
print(f"Saved Merged Complete model to {output_dir / 'merged_complete_model.pkl'}")

with open(output_dir / 'hierarchical_model.pkl', 'wb') as f:
    pickle.dump(model_hier, f)
print(f"Saved Hierarchical model to {output_dir / 'hierarchical_model.pkl'}")

with open(output_dir / 'intermediate_hierarchical_model.pkl', 'wb') as f:
    pickle.dump(model_inter, f)
print(f"Saved Intermediate Hierarchical model to {output_dir / 'intermediate_hierarchical_model.pkl'}")